### Install DependencieS

In [ ]:
!pip install -q torch torchvision timm faiss-cpu tqdm pandas matplotlib scikit-learn pillow kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 43.0 MB/s eta 0:00:00


### Download Stanford Cars Dataset

In [ ]:
import kagglehub
import os
from pathlib import Path
path = kagglehub.dataset_download('jutrera/stanford-car-dataset-by-classes-folder')
DATA_DIR = Path(path)
print(f'Downloaded to: {DATA_DIR}')

100%|██████████| 1.83G/1.83G [00:50<00:00, 39.0MB/s]

Extracting files...


Downloaded to: /root/.cache/kagglehub/datasets/jutrera/stanford-car-dataset-by-classes-folder/versions/2


### Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import random, time, copy
from PIL import Image #Open and process images
from tqdm.notebook import tqdm
from collections import Counter #Count how many times each item appears
import torch
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
import timm #DINO v2 is here
import faiss #Fast similarity search (find closest vectors)
from sklearn.preprocessing import normalize
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


### Explore Dataset Structure (folder tree)

In [ ]:
car_data = DATA_DIR / 'car_data' / 'car_data'#The folder where Kaggle downloaded the dataset
for item in sorted(car_data.iterdir())[:3]:
    print(f' {item.name}/')
    if item.is_dir():#If it is a folder (not a file)
        for sub in sorted(item.iterdir())[:3]:
            imgs = list(sub.glob('*.jpg'))#Find all image files ending with jpg
            print(f'    {sub.name}/ ({len(imgs)} images)')

 test/
    AM General Hummer SUV 2000/ (44 images)
    Acura Integra Type R 2001/ (44 images)
    Acura RL Sedan 2012/ (32 images)
 train/
    AM General Hummer SUV 2000/ (45 images)
    Acura Integra Type R 2001/ (45 images)
    Acura RL Sedan 2012/ (32 images)


### Explore Dataset Structure (folder tree)

In [ ]:
records = []#Creates an empty list
car_data = DATA_DIR / 'car_data' / 'car_data'#dataset folder path

for split in ['train', 'test']:
    split_dir = car_data / split
    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        for img_path in class_dir.glob('*.jpg'):
            records.append({
                'image_path': str(img_path),
                'product_id': class_dir.name,
                'split': split
            })

df_all = pd.DataFrame(records)
print(f'Total images:     {len(df_all)}')
print(f'Total car models: {df_all["product_id"].nunique()}')
counts = df_all['product_id'].value_counts()
print(f'Min images/class: {counts.min()}')
print(f'Max images/class: {counts.max()}')
print(f'Avg images/class: {counts.mean():.1f}')

Total images:     16185
Total car models: 196
Min images/class: 48
Max images/class: 136
Avg images/class: 82.6


### Build DataFrame (image paths, labels, splits)

In [ ]:
unique_products = sorted(df_all['product_id'].unique())
label_map = {pid: i for i, pid in enumerate(unique_products)}
df_all['label'] = df_all['product_id'].map(label_map)
# Use train split as gallery, test split as query
gallery_df = df_all[df_all['split'] == 'train'].reset_index(drop=True)
query_df   = df_all[df_all['split'] == 'test'].reset_index(drop=True)

gallery_paths  = gallery_df['image_path'].tolist()
gallery_labels = gallery_df['label'].values
query_paths    = query_df['image_path'].tolist()
query_labels   = query_df['label'].values

print(f'Gallery: {len(gallery_paths)} images')
print(f'Query:   {len(query_paths)} images')
print(f'Classes: {len(unique_products)}')

Gallery: 8144 images
Query:   8041 images
Classes: 196


### Filter Products (≥2 images)

In [ ]:
print("\n" + "="*50)
print("FILTERING PRODUCTS (≥2 images)")
print("="*50)

product_counts = df_all['product_id'].value_counts()
print(f"\nProducts with 1 image: {(product_counts == 1).sum()}")
print(f"Products with 2 images: {(product_counts == 2).sum()}")
print(f"Products with 3+ images: {(product_counts >= 3).sum()}")
print(f"Max images per product: {product_counts.max()}")

valid_products = product_counts[product_counts >= 2].index
df = df_all[df_all['product_id'].isin(valid_products)].copy()

print(f"\nAfter filtering (≥2 images per product):")
print(f"  Images: {len(df)}")
print(f"  Products: {df['product_id'].nunique()}")
print(f"  Avg images per product: {len(df) / df['product_id'].nunique():.2f}")

# Show products with most images
print("\nTop 5 products with most images:")
print(product_counts.head())


FILTERING PRODUCTS (≥2 images)

Products with 1 image: 0
Products with 2 images: 0
Products with 3+ images: 196
Max images per product: 136

After filtering (≥2 images per product):
  Images: 16185
  Products: 196
  Avg images per product: 82.58

Top 5 products with most images:
product_id
GMC Savana Van 2012                         136
Chrysler 300 SRT-8 2010                      97
Mercedes-Benz 300-Class Convertible 1993     96
Mitsubishi Lancer Sedan 2012                 95
Chevrolet Corvette ZR1 2012                  93
Name: count, dtype: int64


### Label Mapping

In [ ]:
unique_products = sorted(df['product_id'].unique())
label_map = {pid: i for i, pid in enumerate(unique_products)}
df['label'] = df['product_id'].map(label_map)

print(f"\nLabel map created: {len(label_map)} unique products")


Label map created: 196 unique products


### Leak-Safe Retrieval Split (gallery / query)

In [ ]:
print("\n" + "="*50)
print("LEAK-SAFE RETRIEVAL SPLIT")
print("="*50)
RETRIEVAL_SPLIT_SEED = 42

test_df = df[df["split"] == "test"].copy().reset_index(drop=True)
# Keep only classes that have at least 2 test images,
test_counts = test_df["product_id"].value_counts()
valid_test_products = test_counts[test_counts >= 2].index
test_df = test_df[test_df["product_id"].isin(valid_test_products)].copy()

gallery_records = []
query_records = []

for pid in sorted(test_df["product_id"].unique()):
    product_df = (
        test_df[test_df["product_id"] == pid]
        .sample(frac=1.0, random_state=RETRIEVAL_SPLIT_SEED)
        .reset_index(drop=True)
    )
    # Split official TEST images into disjoint gallery/query
    if len(product_df) == 2:
        gallery_records.append(product_df.iloc[:1])
        query_records.append(product_df.iloc[1:])
    else:
        n_gallery = max(1, int(len(product_df) * 0.8))
        n_gallery = min(n_gallery, len(product_df) - 1)  # keep at least 1 query
        gallery_records.append(product_df.iloc[:n_gallery])
        query_records.append(product_df.iloc[n_gallery:])

gallery_df = pd.concat(gallery_records, ignore_index=True)
query_df = pd.concat(query_records, ignore_index=True)

# Safety checks
assert set(gallery_df["split"]) == {"test"}
assert set(query_df["split"]) == {"test"}
assert set(gallery_df["image_path"]).isdisjoint(set(query_df["image_path"])), "Leak: same image in gallery and query"

missing = set(query_df["product_id"]) - set(gallery_df["product_id"])
assert len(missing) == 0, f"Missing classes in gallery: {missing}"

print("✅ No train images are used in retrieval evaluation.")
print("✅ Query images are disjoint from gallery images.")
print(f"\nGallery: {len(gallery_df)} images ({gallery_df['product_id'].nunique()} classes)")
print(f"Query:   {len(query_df)} images ({query_df['product_id'].nunique()} classes)")
print(f"Train images used only for fine-tuning/classification training: {(df_all['split'] == 'train').sum()}")
print(f"Test images used for retrieval eval only: {len(gallery_df) + len(query_df)}")


LEAK-SAFE RETRIEVAL SPLIT
✅ No train images are used in retrieval evaluation.
✅ Query images are disjoint from gallery images.

Gallery: 6355 images (196 classes)
Query:   1686 images (196 classes)
Train images used only for fine-tuning/classification training: 8144
Test images used for retrieval eval only: 8041


## Extract Paths & Labels

In [ ]:
gallery_paths = gallery_df['image_path'].tolist()
gallery_labels = gallery_df['label'].values
gallery_pids = gallery_df['product_id'].tolist()

query_paths = query_df['image_path'].tolist()
query_labels = query_df['label'].values
query_pids = query_df['product_id'].tolist()


### Define Transforms (eval + train augmentation)

In [ ]:
# ── Eval (no augmentation)
eval_tf_224 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_tf_518 = transforms.Compose([
    transforms.Resize((518, 518)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ── Training augmentation
train_aug_224 = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_aug_518 = transforms.Compose([
    transforms.RandomResizedCrop(518, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
# b3T
eval_tf_300 = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_aug_300 = transforms.Compose([
    transforms.RandomResizedCrop(300, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print("✅ Transforms ready.")

✅ Transforms ready.


### Define Retrieval Metrics

In [ ]:
def map_at_k(q_emb, q_labs, g_emb, g_labs, k=5):
    index = faiss.IndexFlatIP(g_emb.shape[1])
    index.add(g_emb.astype("float32"))
    _, I = index.search(q_emb.astype("float32"), k)
    aps = []
    for i, q_lab in enumerate(q_labs):
        hits = (g_labs[I[i]] == q_lab).astype(float)
        if hits.sum() == 0:
            aps.append(0.0); continue
        prec = [hits[:r+1].sum()/(r+1) for r in range(k) if hits[r]]
        aps.append(float(np.mean(prec)))
    return float(np.mean(aps))


def recall_at_k(q_emb, q_labs, g_emb, g_labs, k=5):
    index = faiss.IndexFlatIP(g_emb.shape[1])
    index.add(g_emb.astype("float32"))
    _, I = index.search(q_emb.astype("float32"), k)
    correct = sum(q_labs[i] in g_labs[I[i]] for i in range(len(q_labs)))
    return correct / len(q_labs)


def faiss_latency_ms(q_emb, g_emb, k=5):
    index = faiss.IndexFlatIP(g_emb.shape[1])
    index.add(g_emb.astype("float32"))
    t0 = time.perf_counter()
    index.search(q_emb.astype("float32"), k)
    return (time.perf_counter() - t0) / len(q_emb) * 1000


def storage_mb(emb):
    return (emb.shape[0] * emb.shape[1] * 4) / 1e6


def reset_vram():
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()


def peak_vram_mb():
    if device != "cuda": return 0.0
    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / 1e6

@torch.no_grad()
def extract_embeddings(model, loader, use_amp=False):
    model.eval()
    all_emb, all_labs = [], []
    for imgs, labs in tqdm(loader, desc="Extracting", leave=False):
        imgs = imgs.to(device)
        with torch.amp.autocast('cuda', enabled=use_amp):
            feats = model(imgs).float()
        all_emb.append(feats.cpu().numpy())
        all_labs.extend(labs.numpy())
    emb = normalize(np.vstack(all_emb))
    return emb.astype("float32"), np.array(all_labs)


def run_benchmark(name, model, g_loader, q_loader, use_amp=False):
    print(f"\n📐 Benchmarking {name} ...")
    reset_vram()
    emb_g, labs_g = extract_embeddings(model, g_loader, use_amp)
    emb_q, labs_q = extract_embeddings(model, q_loader, use_amp)
    vram = peak_vram_mb()

    r1  = recall_at_k(emb_q, labs_q, emb_g, labs_g, k=1)
    r5  = recall_at_k(emb_q, labs_q, emb_g, labs_g, k=5)
    mp5 = map_at_k(emb_q, labs_q, emb_g, labs_g, k=5)
    lat = faiss_latency_ms(emb_q, emb_g)
    sto = storage_mb(emb_g)

    benchmark[name] = {
        "Recall@1":       round(r1,  4),
        "Recall@5":       round(r5,  4),
        "mAP@5":          round(mp5, 4),
        "Latency(ms/q)":  round(lat, 4),
        "Storage(MB)":    round(sto, 2),
        "VRAM_peak(MB)":  round(vram, 1),
    }
    print(f"   R@1={r1:.3f}  R@5={r5:.3f}  mAP@5={mp5:.3f}  "
          f"lat={lat:.3f}ms  store={sto:.1f}MB  VRAM={vram:.0f}MB")
    return emb_g, labs_g, emb_q, labs_q


benchmark = {}
print("✅ Metric utils ready.")

✅ Metric utils ready.


### Define RetrievalDataset Class

In [ ]:
class RetrievalDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths, self.labels, self.transform = paths, labels, transform

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img), self.labels[idx]

In [ ]:
TRAIN_BATCH_SIZE = 32
# ── Build eval loaders
g_loader_224 = DataLoader(RetrievalDataset(gallery_paths, gallery_labels, eval_tf_224),
                          batch_size=TRAIN_BATCH_SIZE, shuffle=False, num_workers=2)
q_loader_224 = DataLoader(RetrievalDataset(query_paths,   query_labels,   eval_tf_224),
                          batch_size=TRAIN_BATCH_SIZE, shuffle=False, num_workers=2)

g_loader_518 = DataLoader(RetrievalDataset(gallery_paths, gallery_labels, eval_tf_518),
                          batch_size=16, shuffle=False, num_workers=2)
q_loader_518 = DataLoader(RetrievalDataset(query_paths,   query_labels,   eval_tf_518),
                          batch_size=16, shuffle=False, num_workers=2)

g_loader_300 = DataLoader(RetrievalDataset(gallery_paths, gallery_labels, eval_tf_300),
                          batch_size=32, shuffle=False, num_workers=2)
q_loader_300 = DataLoader(RetrievalDataset(query_paths, query_labels, eval_tf_300),
                          batch_size=32, shuffle=False, num_workers=2)

### #	Zero-Shot

In [ ]:
print("\n" + "="*50)
print("ZERO-SHOT BASELINES")
print("="*50)


# ── ResNet50
resnet_base = models.resnet50(weights="DEFAULT")
resnet_base.fc = nn.Identity()
resnet_base = resnet_base.to(device).eval()
run_benchmark("ResNet50 (zero-shot)", resnet_base, g_loader_224, q_loader_224)
del resnet_base; torch.cuda.empty_cache()

# ── Efficientnet B3
effnet_b3_base = models.efficientnet_b3(weights="DEFAULT")
effnet_b3_base.classifier = nn.Identity()
effnet_b3_base = effnet_b3_base.to(device).eval()
run_benchmark("EfficientNet-B3 (zero-shot)", effnet_b3_base, g_loader_300, q_loader_300)
del effnet_b3_base; torch.cuda.empty_cache()

# ── Swin-Tiny
swin_base = timm.create_model("swin_tiny_patch4_window7_224",
                               pretrained=True, num_classes=0)
swin_base = swin_base.to(device).eval()
run_benchmark("Swin-Tiny (zero-shot)", swin_base, g_loader_224, q_loader_224)
del swin_base; torch.cuda.empty_cache()

print("\n✅ Zero-shot baselines done.")

# ── DINOv2
dino_base = timm.create_model("vit_base_patch14_dinov2.lvd142m",
                               pretrained=True, num_classes=0)
dino_base = dino_base.to(device).eval()
run_benchmark("DINOv2 (zero-shot)", dino_base, g_loader_518, q_loader_518)
del dino_base; torch.cuda.empty_cache()

print("\n" + "="*60)
print("ZERO-SHOT BASELINE RESULTS")
print("="*60)


zs_keys = [k for k in benchmark.keys() if "zero-shot" in k]
zs_df = pd.DataFrame({k: benchmark[k] for k in zs_keys}).T
zs_df.index.name = "Model"

cols = ["Recall@1", "Recall@5", "mAP@5", "Latency(ms/q)", "Storage(MB)", "VRAM_peak(MB)"]
zs_df = zs_df[[c for c in cols if c in zs_df.columns]]
styled = zs_df.style \
    .set_caption("🔬 Zero-Shot Image Retrieval Benchmark") \
    .format({
        "Recall@1": "{:.3f}",
        "Recall@5": "{:.3f}",
        "mAP@5": "{:.3f}",
        "Latency(ms/q)": "{:.4f}",
        "Storage(MB)": "{:.2f}",
        "VRAM_peak(MB)": "{:.1f}"
    }) \
    .background_gradient(subset=["Recall@1", "Recall@5", "mAP@5"], cmap="Greens", vmin=0, vmax=1) \
    .background_gradient(subset=["Latency(ms/q)", "Storage(MB)", "VRAM_peak(MB)"], cmap="Reds") \
    .set_properties(**{"text-align": "center"}) \
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "center"), ("font-weight", "bold"), ("background-color", "#f0f0f0")]},
        {"selector": "caption", "props": [("font-size", "16px"), ("font-weight", "bold"), ("padding", "10px")]}
    ])

display(styled)
print("="*60)


ZERO-SHOT BASELINES
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 189MB/s]



📐 Benchmarking ResNet50 (zero-shot) ...


Extracting:   0%|          | 0/199 [00:00<?, ?it/s]

Extracting:   0%|          | 0/53 [00:00<?, ?it/s]

...


📐 Benchmarking DINOv2 (zero-shot) ...


Extracting:   0%|          | 0/398 [00:00<?, ?it/s]

Extracting:   0%|          | 0/106 [00:00<?, ?it/s]

   R@1=0.676  R@5=0.855  mAP@5=0.711  lat=0.087ms  store=19.5MB  VRAM=1284MB

ZERO-SHOT BASELINE RESULTS


,Recall@1,Recall@5,mAP@5,Latency(ms/q),Storage(MB),VRAM_peak(MB)
Model,,,,,,
ResNet50 (zero-shot),0.240,0.490,0.317,0.2278,52.06,447.8
EfficientNet-B3 (zero-shot),0.317,0.601,0.403,0.2798,39.05,977.6
Swin-Tiny (zero-shot),0.258,0.499,0.330,0.0922,19.52,623.4
DINOv2 (zero-shot),0.676,0.855,0.711,0.0869,19.52,1283.7


Define TripletDataset Class

In [ ]:
class TripletDataset(Dataset):
    def __init__(self, df, transform, model=None, hard_neg_every=5):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.model = model
        self.hard_neg_every = hard_neg_every  # how often to use hard negatives
        self.call_count = 0

        self.label_to_idx = {}
        for i, row in self.df.iterrows():
            self.label_to_idx.setdefault(row['label'], []).append(i)
        self.valid_labels = [l for l, v in self.label_to_idx.items() if len(v) >= 2]

        # Cache for hard negatives
        self.hard_neg_cache = {}  #list of hard negative indices

    def __len__(self):
        return len(self.df)

    def _load(self, idx):
        return Image.open(self.df.loc[idx, 'image_path']).convert('RGB')

    def _random_negative_idx(self, anchor_label):
        neg_label = random.choice([l for l in self.valid_labels if l != anchor_label])
        return random.choice(self.label_to_idx[neg_label])

    def _hard_negative_idx(self, anchor_label):
        if anchor_label not in self.hard_neg_cache:
            return self._random_negative_idx(anchor_label)
        return random.choice(self.hard_neg_cache[anchor_label])

    def __getitem__(self, _):
        anchor_label = random.choice(self.valid_labels)
        a_idx, p_idx = random.sample(self.label_to_idx[anchor_label], 2)
        self.call_count += 1
        if self.model is not None and self.call_count % self.hard_neg_every == 0:
            n_idx = self._hard_negative_idx(anchor_label)
        else:
            n_idx = self._random_negative_idx(anchor_label)

        anchor   = self.transform(self._load(a_idx))
        positive = self.transform(self._load(p_idx))
        negative = self.transform(self._load(n_idx))

        return anchor, positive, negative, anchor_label


@torch.no_grad()
def refresh_hard_negatives(model, dataset, top_k=10, sample_per_class=5):
    print("  🔍 Refreshing hard negative cache...")
    model.eval()
    all_emb, all_labels, all_indices = [], [], []

    sampled_indices = []
    for label, idxs in dataset.label_to_idx.items():
        sampled_indices.extend(random.sample(idxs, min(sample_per_class, len(idxs))))

    for idx in sampled_indices:
        img = dataset.transform(dataset._load(idx)).unsqueeze(0).to(device)
        feat = model(img).cpu().numpy()
        all_emb.append(feat)
        all_labels.append(dataset.df.loc[idx, 'label'])
        all_indices.append(idx)

    all_emb = normalize(np.vstack(all_emb)).astype("float32")
    all_labels = np.array(all_labels)
    all_indices = np.array(all_indices)

    index = faiss.IndexFlatIP(all_emb.shape[1])
    index.add(all_emb)
    _, I = index.search(all_emb, top_k + 1)

    hard_neg_cache = {}
    for i, label in enumerate(all_labels):
        hard_negs = []
        for j in I[i][1:]:
            if all_labels[j] != label:
                hard_negs.append(all_indices[j])
        if hard_negs:
            hard_neg_cache.setdefault(label, []).extend(hard_negs)

    dataset.hard_neg_cache = hard_neg_cache
    print(f"  ✅ Hard negative cache built for {len(hard_neg_cache)} classes.")
    model.train()

### Define TripletEmbeddingModel

In [ ]:
class TripletEmbeddingModel(nn.Module):
    def __init__(self, backbone, backbone_out_dim=None, embed_dim=None):
        super().__init__()
        self.backbone = backbone

    def forward(self, x):
        feat = self.backbone(x)
        return F.normalize(feat, p=2, dim=1)

    def extract_features(self, x):
        return self.forward(x)


def freeze_backbone(model):
    for p in model.backbone.parameters():
        p.requires_grad = False


def unfreeze_last_layers(model, n_layers=2, model_type="resnet"):
    for p in model.backbone.parameters():
        p.requires_grad = False

    if model_type == "resnet":
        layers = [model.backbone.layer4, model.backbone.layer3]
        for layer in layers[:n_layers]:
            for p in layer.parameters():
                p.requires_grad = True

    elif model_type == "efficientnet":
        blocks = list(model.backbone.features.children())
        for block in blocks[-n_layers:]:
            for p in block.parameters():
                p.requires_grad = True

    elif model_type == "dino":
        blocks = list(model.backbone.blocks)
        for block in blocks[-n_layers:]:
            for p in block.parameters():
                p.requires_grad = True
        if hasattr(model.backbone, "norm"):
            for p in model.backbone.norm.parameters():
                p.requires_grad = True

    elif model_type == "swin":
        layers = [model.backbone.layers[i] for i in range(len(model.backbone.layers))]
        for layer in layers[-n_layers:]:
            for p in layer.parameters():
                p.requires_grad = True
        if hasattr(model.backbone, "norm"):
            for p in model.backbone.norm.parameters():
                p.requires_grad = True


def count_trainable(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


print("✅ TripletEmbeddingModel ready.")

✅ TripletEmbeddingModel ready.


### Set Training / Define Freeze & Unfreeze Utils

In [ ]:
TRIPLET_MARGIN   = 0.5
FINETUNE_EPS     = 10     # was 5
BACKBONE_LR      = 1e-3 #1e-3
WEIGHT_DECAY     = 1e-4
TRAIN_BATCH_SIZE = 32
triplet_criterion = nn.TripletMarginLoss(margin=TRIPLET_MARGIN, p=2)


def make_optimizer(model):
    backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": BACKBONE_LR})
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)


def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss, n = 0.0, 0
    for anchor, positive, negative, _ in tqdm(loader, desc="Training", leave=False):
        anchor   = anchor.to(device)
        positive = positive.to(device)
        negative = negative.to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=(device == "cuda")):
            a_emb = model(anchor)
            p_emb = model(positive)
            n_emb = model(negative)
            loss  = triplet_criterion(a_emb, p_emb, n_emb)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * anchor.size(0)
        n += anchor.size(0)
    return total_loss / n


def save_checkpoint(state_dict, save_path):
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    torch.save(state_dict, save_path)


def train_model(model, triplet_loader, model_type="resnet", name="Model", save_dir="checkpoints"):
    save_path = os.path.join(save_dir, f"{name.replace(' ', '_')}_best.pth")
    scaler = torch.amp.GradScaler('cuda', enabled=(device == "cuda"))
    history = []
    best_loss, best_state = float("inf"), None

    print(f"\n{'='*50}")
    print(f"{name} — backbone fine-tune ({FINETUNE_EPS} epochs)")
    unfreeze_last_layers(model, n_layers=2, model_type=model_type)
    optimizer = make_optimizer(model)
    total, trainable = count_trainable(model)
    print(f"Trainable: {trainable:,} / {total:,}")

    for ep in range(1, FINETUNE_EPS + 1):
        refresh_hard_negatives(model, triplet_loader.dataset)
        triplet_loader.dataset.model = model

        loss = train_one_epoch(model, triplet_loader, optimizer, scaler)
        history.append({"epoch": ep, "loss": loss})
        if loss < best_loss:
            best_loss = loss
            best_state = copy.deepcopy(model.state_dict())
            save_checkpoint(best_state, save_path)
            print(f"  Epoch {ep}/{FINETUNE_EPS} | loss {loss:.4f} 🔥 BEST → saved")
        else:
            print(f"  Epoch {ep}/{FINETUNE_EPS} | loss {loss:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"✅ {name} done. Best loss: {best_loss:.4f}")
    print(f"💾 Checkpoint: {save_path}")
    return pd.DataFrame(history)


print("✅ Training utils ready.")

✅ Training utils ready.


## #	Fine-Tune: ResNet50 (Triplet)

In [ ]:
print("\n" + "="*50)
print("FINE-TUNING ResNet50 WITH TRIPLET LOSS")
print("="*50)

train_df = df[df["split"] == "train"].copy()

resnet_triplet_ds     = TripletDataset(train_df, train_aug_224)
resnet_triplet_loader = DataLoader(resnet_triplet_ds, batch_size=TRAIN_BATCH_SIZE,
                                   shuffle=True, num_workers=2, drop_last=True)
resnet_backbone = models.resnet50(weights="DEFAULT")
resnet_backbone.fc = nn.Identity()
resnet_ft = TripletEmbeddingModel(resnet_backbone).to(device)

resnet_history = train_model(resnet_ft, resnet_triplet_loader,
                              model_type="resnet", name="ResNet50")

@torch.no_grad()
def extract_ft(model, loader):
    model.eval()
    all_emb, all_labs = [], []
    for imgs, labs in tqdm(loader, leave=False):
        imgs = imgs.to(device)
        with torch.amp.autocast('cuda', enabled=(device=="cuda")):
            feats = model.extract_features(imgs).float()
        all_emb.append(F.normalize(feats, dim=-1).cpu().numpy())
        all_labs.extend(labs.numpy())
    emb = np.vstack(all_emb)
    return normalize(emb).astype("float32"), np.array(all_labs)

reset_vram()
emb_g_rn, labs_g_rn = extract_ft(resnet_ft, g_loader_224)
emb_q_rn, labs_q_rn = extract_ft(resnet_ft, q_loader_224)
vram_rn = peak_vram_mb()

benchmark["ResNet50 (triplet FT)"] = {
    "Recall@1":      round(recall_at_k(emb_q_rn, labs_q_rn, emb_g_rn, labs_g_rn, k=1), 4),
    "Recall@5":      round(recall_at_k(emb_q_rn, labs_q_rn, emb_g_rn, labs_g_rn, k=5), 4),
    "mAP@5":         round(map_at_k(emb_q_rn, labs_q_rn, emb_g_rn, labs_g_rn, k=5), 4),
    "Latency(ms/q)": round(faiss_latency_ms(emb_q_rn, emb_g_rn), 4),
    "Storage(MB)":   round(storage_mb(emb_g_rn), 2),
    "VRAM_peak(MB)": round(vram_rn, 1),
}
print("\n✅ ResNet50 FT benchmark stored.")
print(benchmark["ResNet50 (triplet FT)"])


FINE-TUNING ResNet50 WITH TRIPLET LOSS
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 161MB/s]



ResNet50 — backbone fine-tune (10 epochs)
Trainable: 22,063,104 / 23,508,032
  🔍 Refreshing hard negative cache...


KeyboardInterrupt: 

### loss

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

epochs = resnet_history["epoch"].tolist()
losses = resnet_history["loss"].tolist()
best_epoch = epochs[losses.index(min(losses))]
best_loss  = min(losses)

ax.plot(epochs, losses,
        color="#4C72B0", marker="o",
        linewidth=2, markersize=5,
        label="ResNet50 (triplet FT)")

ax.scatter([best_epoch], [best_loss], color="red", zorder=5, s=80)
ax.axvline(x=best_epoch, color="red", linestyle="--", alpha=0.5)
ax.annotate(f"  best: {best_loss:.4f}",
            xy=(best_epoch, best_loss),
            fontsize=9, color="red")

ax.set_xlabel("Epoch")
ax.set_ylabel("Triplet Loss")
ax.set_title("ResNet50 — Training Loss")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("loss_resnet50.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Epochs: {len(epochs)} | Best loss: {best_loss:.4f} at epoch {best_epoch}")

### Fine-Tune: EfficientNet-B3 (Triplet)

In [ ]:
print("\n" + "="*50)
print("FINE-TUNING EfficientNet-B3 WITH TRIPLET LOSS")
print("="*50)

effnet_b3_triplet_ds     = TripletDataset(train_df, train_aug_300)
effnet_b3_triplet_loader = DataLoader(effnet_b3_triplet_ds, batch_size=32,
                                      shuffle=True, num_workers=2, drop_last=True)

effnet_b3_backbone = models.efficientnet_b3(weights="DEFAULT")
effnet_b3_backbone.classifier = nn.Identity()
effnet_b3_ft = TripletEmbeddingModel(effnet_b3_backbone).to(device)

effnet_b3_history = train_model(effnet_b3_ft, effnet_b3_triplet_loader,
                                 model_type="efficientnet", name="EfficientNet-B3")

@torch.no_grad()
def extract_ft(model, loader):
    model.eval()
    all_emb, all_labs = [], []
    for imgs, labs in tqdm(loader, leave=False):
        imgs = imgs.to(device)
        with torch.amp.autocast('cuda', enabled=(device=="cuda")):
            feats = model.extract_features(imgs).float()
        all_emb.append(F.normalize(feats, dim=-1).cpu().numpy())
        all_labs.extend(labs.numpy())
    emb = np.vstack(all_emb)
    return normalize(emb).astype("float32"), np.array(all_labs)
reset_vram()
emb_g_b3, labs_g_b3 = extract_ft(effnet_b3_ft, g_loader_300)
emb_q_b3, labs_q_b3 = extract_ft(effnet_b3_ft, q_loader_300)
vram_b3 = peak_vram_mb()

benchmark["EfficientNet-B3 (triplet FT)"] = {
    "Recall@1":      round(recall_at_k(emb_q_b3, labs_q_b3, emb_g_b3, labs_g_b3, k=1), 4),
    "Recall@5":      round(recall_at_k(emb_q_b3, labs_q_b3, emb_g_b3, labs_g_b3, k=5), 4),
    "mAP@5":         round(map_at_k(emb_q_b3, labs_q_b3, emb_g_b3, labs_g_b3, k=5), 4),
    "Latency(ms/q)": round(faiss_latency_ms(emb_q_b3, emb_g_b3), 4),
    "Storage(MB)":   round(storage_mb(emb_g_b3), 2),
    "VRAM_peak(MB)": round(vram_b3, 1),
}
print("\n✅ EfficientNet-B3 FT benchmark stored.")
print(benchmark["EfficientNet-B3 (triplet FT)"])


FINE-TUNING EfficientNet-B3 WITH TRIPLET LOSS
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 160MB/s]



EfficientNet-B3 — backbone fine-tune (10 epochs)
Trainable: 3,877,114 / 10,696,232
  🔍 Refreshing hard negative cache...
  ✅ Hard negative cache built for 196 classes.


Training:   0%|          | 0/254 [00:00<?, ?it/s]

  Epoch 1/10 | loss 0.3135 🔥 BEST → saved
  🔍 Refreshing hard negative cache...
  ✅ Hard negative cache built for 196 classes.


...

Training:   0%|          | 0/254 [00:00<?, ?it/s]

  Epoch 10/10 | loss 0.1646 🔥 BEST → saved
✅ EfficientNet-B3 done. Best loss: 0.1646
💾 Checkpoint: checkpoints/EfficientNet-B3_best.pth


  0%|          | 0/199 [00:00<?, ?it/s]

  0%|          | 0/53 [00:00<?, ?it/s]


✅ EfficientNet-B3 FT benchmark stored.
{'Recall@1': 0.3304, 'Recall@5': 0.6311, 'mAP@5': 0.4134, 'Latency(ms/q)': 0.3399, 'Storage(MB)': 39.05, 'VRAM_peak(MB)': 640.9}


### loss

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

epochs = effnet_b3_history["epoch"].tolist()
losses = effnet_b3_history["loss"].tolist()
best_epoch = epochs[losses.index(min(losses))]
best_loss  = min(losses)

ax.plot(epochs, losses,
        color="#DD8452", marker="o",
        linewidth=2, markersize=5,
        label="EfficientNet-B3 (triplet FT)")

ax.scatter([best_epoch], [best_loss], color="red", zorder=5, s=80)
ax.axvline(x=best_epoch, color="red", linestyle="--", alpha=0.5)
ax.annotate(f"  best: {best_loss:.4f}",
            xy=(best_epoch, best_loss),
            fontsize=9, color="red")

ax.set_xlabel("Epoch")
ax.set_ylabel("Triplet Loss")
ax.set_title("EfficientNet-B3 — Training Loss")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("loss_effnet_b3.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Epochs: {len(epochs)} | Best loss: {best_loss:.4f} at epoch {best_epoch}")

### Fine-Tune: DINOv2 (Triplet)

In [ ]:
print("\n" + "="*50)
print("FINE-TUNING DINOv2 WITH TRIPLET LOSS")
print("="*50)

dino_triplet_ds     = TripletDataset(train_df, train_aug_518)
dino_triplet_loader = DataLoader(dino_triplet_ds, batch_size=16,
                                 shuffle=True, num_workers=2, drop_last=True)

dino_backbone = timm.create_model("vit_base_patch14_dinov2.lvd142m",
                                   pretrained=True, num_classes=0)
dino_ft = TripletEmbeddingModel(dino_backbone, backbone_out_dim=768, embed_dim=256).to(device)

dino_history = train_model(dino_ft, dino_triplet_loader,
                            model_type="dino", name="DINOv2")
@torch.no_grad()
def extract_ft(model, loader):
    model.eval()
    all_emb, all_labs = [], []
    for imgs, labs in tqdm(loader, leave=False):
        imgs = imgs.to(device)
        with torch.amp.autocast('cuda', enabled=(device=="cuda")):
            feats = model.extract_features(imgs).float()
        all_emb.append(F.normalize(feats, dim=-1).cpu().numpy())
        all_labs.extend(labs.numpy())
    emb = np.vstack(all_emb)
    return normalize(emb).astype("float32"), np.array(all_labs)

reset_vram()
emb_g_dino, labs_g_dino = extract_ft(dino_ft, g_loader_518)
emb_q_dino, labs_q_dino = extract_ft(dino_ft, q_loader_518)
vram_dino = peak_vram_mb()

benchmark["DINOv2 (triplet FT)"] = {
    "Recall@1":      round(recall_at_k(emb_q_dino, labs_q_dino, emb_g_dino, labs_g_dino, k=1), 4),
    "Recall@5":      round(recall_at_k(emb_q_dino, labs_q_dino, emb_g_dino, labs_g_dino, k=5), 4),
    "mAP@5":         round(map_at_k(emb_q_dino, labs_q_dino, emb_g_dino, labs_g_dino, k=5), 4),
    "Latency(ms/q)": round(faiss_latency_ms(emb_q_dino, emb_g_dino), 4),
    "Storage(MB)":   round(storage_mb(emb_g_dino), 2),
    "VRAM_peak(MB)": round(vram_dino, 1),
}
print("\n✅ DINOv2 FT benchmark stored.")
print(benchmark["DINOv2 (triplet FT)"])

### loss

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

epochs = dino_history["epoch"].tolist()
losses = dino_history["loss"].tolist()
best_epoch = epochs[losses.index(min(losses))]
best_loss  = min(losses)

ax.plot(epochs, losses,
        color="#C44E52", marker="o",
        linewidth=2, markersize=5,
        label="DINOv2 (triplet FT)")

ax.scatter([best_epoch], [best_loss], color="red", zorder=5, s=80)
ax.axvline(x=best_epoch, color="red", linestyle="--", alpha=0.5)
ax.annotate(f"  best: {best_loss:.4f}",
            xy=(best_epoch, best_loss),
            fontsize=9, color="red")

ax.set_xlabel("Epoch")
ax.set_ylabel("Triplet Loss")
ax.set_title("DINOv2 — Training Loss")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("loss_dino.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Epochs: {len(epochs)} | Best loss: {best_loss:.4f} at epoch {best_epoch}")

### Fine-Tune: Swin-Tiny (Triplet)

In [ ]:
print("\n" + "="*50)
print("FINE-TUNING Swin-Tiny WITH TRIPLET LOSS")
print("="*50)

swin_triplet_ds     = TripletDataset(train_df, train_aug_224)
swin_triplet_loader = DataLoader(swin_triplet_ds, batch_size=TRAIN_BATCH_SIZE,
                                 shuffle=True, num_workers=2, drop_last=True)

swin_backbone = timm.create_model("swin_tiny_patch4_window7_224",
                                   pretrained=True, num_classes=0)
swin_ft = TripletEmbeddingModel(swin_backbone, backbone_out_dim=768, embed_dim=256).to(device)

swin_history = train_model(swin_ft, swin_triplet_loader,
                            model_type="swin", name="Swin-Tiny")
@torch.no_grad()
def extract_ft(model, loader):
    model.eval()
    all_emb, all_labs = [], []
    for imgs, labs in tqdm(loader, leave=False):
        imgs = imgs.to(device)
        with torch.amp.autocast('cuda', enabled=(device=="cuda")):
            feats = model.extract_features(imgs).float()
        all_emb.append(F.normalize(feats, dim=-1).cpu().numpy())
        all_labs.extend(labs.numpy())
    emb = np.vstack(all_emb)
    return normalize(emb).astype("float32"), np.array(all_labs)

reset_vram()
emb_g_swin, labs_g_swin = extract_ft(swin_ft, g_loader_224)
emb_q_swin, labs_q_swin = extract_ft(swin_ft, q_loader_224)
vram_swin = peak_vram_mb()

benchmark["Swin-Tiny (triplet FT)"] = {
    "Recall@1":      round(recall_at_k(emb_q_swin, labs_q_swin, emb_g_swin, labs_g_swin, k=1), 4),
    "Recall@5":      round(recall_at_k(emb_q_swin, labs_q_swin, emb_g_swin, labs_g_swin, k=5), 4),
    "mAP@5":         round(map_at_k(emb_q_swin, labs_q_swin, emb_g_swin, labs_g_swin, k=5), 4),
    "Latency(ms/q)": round(faiss_latency_ms(emb_q_swin, emb_g_swin), 4),
    "Storage(MB)":   round(storage_mb(emb_g_swin), 2),
    "VRAM_peak(MB)": round(vram_swin, 1),
}
print("\n✅ Swin-Tiny FT benchmark stored.")
print(benchmark["Swin-Tiny (triplet FT)"])

### loss

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

epochs = swin_history["epoch"].tolist()
losses = swin_history["loss"].tolist()
best_epoch = epochs[losses.index(min(losses))]
best_loss  = min(losses)

ax.plot(epochs, losses,
        color="#55A868", marker="o",
        linewidth=2, markersize=5,
        label="Swin-Tiny (triplet FT)")

ax.scatter([best_epoch], [best_loss], color="red", zorder=5, s=80)
ax.axvline(x=best_epoch, color="red", linestyle="--", alpha=0.5)
ax.annotate(f"  best: {best_loss:.4f}",
            xy=(best_epoch, best_loss),
            fontsize=9, color="red")

ax.set_xlabel("Epoch")
ax.set_ylabel("Triplet Loss")
ax.set_title("Swin-Tiny — Training Loss")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("loss_swin.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Epochs: {len(epochs)} | Best loss: {best_loss:.4f} at epoch {best_epoch}")

### Benchmark Results

In [ ]:
print("\n" + "="*60)
print("FULL BENCHMARK — ALL MODELS")
print("="*60)

df_bench = pd.DataFrame(benchmark).T
df_bench.index.name = "Model"
display(df_bench)
models_list = list(benchmark.keys())

base_colors = {
    "ResNet50":      "#4C72B0",
    "EfficientNet":  "#DD8452",
    "Swin":          "#55A868",
    "DINOv2":        "#C44E52",
}

def get_color(name):
    for key, color in base_colors.items():
        if key.lower() in name.lower():
            return color
    return "#999999"

def get_hatch(name):
    if "zero-shot" in name:
        return ""
    return "//"

colors  = [get_color(m) for m in models_list]
hatches = [get_hatch(m) for m in models_list]
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle("Benchmark — Stanford Cars Image Retrieval", fontsize=16, fontweight="bold")

x = np.arange(len(models_list))
w = 0.25

ax = axes[0, 0]
for i, metric in enumerate(["Recall@1", "Recall@5", "mAP@5"]):
    vals = [benchmark[m][metric] for m in models_list]
    ax.bar(x + (i-1)*w, vals, w, label=metric)
ax.set_xticks(x)
ax.set_xticklabels(models_list, rotation=25, ha="right", fontsize=8)
ax.set_ylabel("Score"); ax.set_ylim(0, 1)
ax.set_title("Retrieval Quality Metrics")
ax.legend(); ax.grid(axis="y", alpha=0.3)

ax = axes[0, 1]
map_vals = [benchmark[m]["mAP@5"] for m in models_list]
bars = ax.barh(models_list, map_vals, color=colors)
for bar, h, val in zip(bars, hatches, map_vals):
    bar.set_hatch(h)
    ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=9)
ax.set_xlabel("mAP@5"); ax.set_xlim(0, 1)
ax.set_title("Mean Average Precision @ 5")
ax.grid(axis="x", alpha=0.3)

ax = axes[1, 0]
lat_vals = [benchmark[m]["Latency(ms/q)"] for m in models_list]
bars = ax.bar(models_list, lat_vals, color=colors)
for bar, h, val in zip(bars, hatches, lat_vals):
    bar.set_hatch(h)
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.0001,
            f"{val:.3f}", ha="center", va="bottom", fontsize=8)
ax.set_ylabel("ms / query"); ax.set_title("FAISS Search Latency")
ax.set_xticks(x)
ax.set_xticklabels(models_list, rotation=25, ha="right", fontsize=8)
ax.grid(axis="y", alpha=0.3)

ax = axes[1, 1]
sto  = [benchmark[m]["Storage(MB)"]   for m in models_list]
vram = [benchmark[m]["VRAM_peak(MB)"] for m in models_list]
map5 = [benchmark[m]["mAP@5"]         for m in models_list]
sc = ax.scatter(sto, vram, s=[v*2000 for v in map5],
                c=colors, alpha=0.75,
                edgecolors="white", linewidth=1.5)
for i, m in enumerate(models_list):
    ax.annotate(m, (sto[i], vram[i]),
                textcoords="offset points", xytext=(6, 4), fontsize=7)
ax.set_xlabel("Gallery Storage (MB)")
ax.set_ylabel("Peak VRAM (MB)")
ax.set_title("Storage vs VRAM  (bubble size ∝ mAP@5)")
ax.grid(alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="gray", label="Zero-shot"),
    Patch(facecolor="gray", hatch="//", label="Fine-tuned"),
]
fig.legend(handles=legend_elements, loc="lower center",
           ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig("benchmark.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n✅ Saved benchmark.png")
print("✅ Full benchmark dict: benchmark")